# ARC NeuroGolf static ONNX solver

Reference layout adapted from the uploaded fill/additive-marking notebook. The task-specific modelling cell uses a semantic feature-tree or a symbolic reflection builder, not raw output-template lookup.

In [1]:
!rm -rf /kaggle/working/*
%reset -f

In [2]:
COMPETITION = '/kaggle/input/competitions/neurogolf-2026'

In [3]:
import importlib.util, subprocess, sys
missing=[p for p in ['onnx','onnxruntime','onnxscript','torch','numpy'] if importlib.util.find_spec(p) is None]
if missing:
    subprocess.check_call([sys.executable,'-m','pip','install','-q',*missing])
print('dependencies ok')

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.7/18.7 MB 74.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 722.0/722.0 kB 27.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 166.8/166.8 kB 7.0 MB/s eta 0:00:00
dependencies ok


In [4]:
import json, os, time, hashlib, zipfile,  csv, base64
import glob, sys,math, random, collections,io,shutil
from pathlib import Path
import numpy as np
import torch
import onnx
import onnxruntime as ort
import torch, torch.nn as nn, torch.nn.functional as F
from collections import defaultdict
from onnx import shape_inference

In [5]:
TASK_ID="task212"
H=W=30
CH=10
TASK_PATH=Path(COMPETITION)/"task212.json"
if not TASK_PATH.exists():
    TASK_PATH=Path("/mnt/data/task212.json")
OUT=Path(".")

In [6]:
class Task212CanvasMask(nn.Module):
    def __init__(self):
        super().__init__()
        rows=torch.arange(H,dtype=torch.float32)
        r=rows[:,None]; s=rows[None,:]
        self.register_buffer('rel_le', (r <= s).float())
        self.register_buffer('rel_ge', (r >= s).float())
        self.register_buffer('row_idx', rows.view(H,1))
        self.register_buffer('eps', torch.tensor(1e-6, dtype=torch.float32))
    def forward(self, x):
        active=(x.sum(dim=1)>0.0).float()
        m1=x[:,1,:,:]*active; m2=x[:,2,:,:]*active; m5=x[:,5,:,:]*active
        row_scores=m5.sum(dim=2)
        denom=row_scores.sum(dim=1, keepdim=True)+self.eps.view(1,1)
        sep_idx=(row_scores*self.row_idx.view(1,H)).sum(dim=1, keepdim=True)/denom
        rows=self.row_idx.view(1,H,1)
        tgt_above=(rows < sep_idx.view(1,1,1)).float(); tgt_below=(rows > sep_idx.view(1,1,1)).float()
        c1_above=torch.matmul(self.rel_le, m1*tgt_above)*tgt_above
        c1_below=torch.matmul(self.rel_ge, m1*tgt_below)*tgt_below
        gen1=((c1_above+c1_below)>0.0).float()*active
        c2_above=torch.matmul(self.rel_ge, m2*tgt_above)*tgt_above
        c2_below=torch.matmul(self.rel_le, m2*tgt_below)*tgt_below
        gen2=((c2_above+c2_below)>0.0).float()*active
        out5=(m5>0.5).float()*active
        out1=((m1+gen1)>0.0).float()*active*(1.0-out5)
        out2=((m2+gen2)>0.0).float()*active*(1.0-out5)*(1.0-out1)
        occ=((out1+out2+out5)>0.0).float()
        out0=(1.0-occ)*active
        z=torch.zeros_like(out0)
        return torch.cat([out0.unsqueeze(1),out1.unsqueeze(1),out2.unsqueeze(1),z.unsqueeze(1),z.unsqueeze(1),out5.unsqueeze(1),z.unsqueeze(1),z.unsqueeze(1),z.unsqueeze(1),z.unsqueeze(1)], dim=1)

def grid_to_tensor(grid):
    arr=np.array(grid,dtype=np.int64); x=np.zeros((1,CH,H,W),dtype=np.float32)
    h,w=arr.shape
    for c in range(CH): x[0,c,:h,:w]=(arr==c)
    return x

def output_to_tensor(grid):
    return grid_to_tensor(grid)

def pred_grid(y):
    return np.asarray(y)[0].argmax(axis=0).astype(np.int64)

def tensor_exact(y, exp_grid):
    return np.allclose(np.asarray(y), output_to_tensor(exp_grid), atol=1e-5)


In [7]:
data=json.load(open(TASK_PATH))
model=Task212CanvasMask().eval()

# Export ONNX
onnx_path=Path(f"{TASK_ID}.onnx")
dummy=torch.from_numpy(grid_to_tensor(data['test'][0]['input']))
torch.onnx.export(model, dummy, onnx_path.as_posix(), input_names=['input'], output_names=['output'], opset_version=17, do_constant_folding=True, dynamo=False)

m=onnx.load(onnx_path.as_posix())
m=shape_inference.infer_shapes(m)
onnx.save(m, onnx_path.as_posix())
onnx.checker.check_model(m)
ops=sorted(set(n.op_type for n in m.graph.node))
print('ops', ops)
print('size', onnx_path.stat().st_size)

/tmp/ipykernel_16/1630972378.py:7: DeprecationWarning: You are using the legacy TorchScript-based ONNX export. Starting in PyTorch 2.9, the new torch.export-based ONNX exporter has become the default. Learn more about the new export logic: https://docs.pytorch.org/docs/stable/onnx_export.html. For exporting control flow: https://pytorch.org/tutorials/beginner/onnx/export_control_flow_model_to_onnx_tutorial.html
  torch.onnx.export(model, dummy, onnx_path.as_posix(), input_names=['input'], output_names=['output'], opset_version=17, do_constant_folding=True, dynamo=False)


ops ['Add', 'Cast', 'Concat', 'Constant', 'Div', 'Gather', 'Greater', 'Less', 'MatMul', 'Mul', 'ReduceSum', 'Reshape', 'Sub', 'Unsqueeze']
size 21313


In [8]:
sess=ort.InferenceSession(onnx_path.as_posix(), providers=['CPUExecutionProvider'])
summary={}
for split in ['train','test','arc-gen']:
    ok=tensor_ok=zero_ok=0
    for ex in data[split]:
        x=grid_to_tensor(ex['input'])
        y=sess.run(None, {'input':x})[0]
        tensor_ok += int(tensor_exact(y, ex['output']))
        active=(x.sum(axis=1)[0]>0)
        zero_ok += int(np.allclose(y[0,:,~active], 0.0, atol=1e-5))
    summary[f'{split}_tensor_exact_zero_padded']={'ok':tensor_ok,'total':len(data[split])}
    summary[f'{split}_outside_active_all_channels_zero']={'ok':zero_ok,'total':len(data[split])}
print(json.dumps(summary, indent=2))
assert summary['train_tensor_exact_zero_padded']['ok']==len(data['train'])
assert summary['test_tensor_exact_zero_padded']['ok']==len(data['test'])
assert summary['arc-gen_tensor_exact_zero_padded']['ok']==len(data['arc-gen'])

{
  "train_tensor_exact_zero_padded": {
    "ok": 2,
    "total": 2
  },
  "train_outside_active_all_channels_zero": {
    "ok": 2,
    "total": 2
  },
  "test_tensor_exact_zero_padded": {
    "ok": 1,
    "total": 1
  },
  "test_outside_active_all_channels_zero": {
    "ok": 1,
    "total": 1
  },
  "arc-gen_tensor_exact_zero_padded": {
    "ok": 262,
    "total": 262
  },
  "arc-gen_outside_active_all_channels_zero": {
    "ok": 262,
    "total": 262
  }
}


In [9]:
forbidden={'Loop','Scan','NonZero','Unique','Script','Function'}
empty=[(n.name,n.op_type,list(n.input)) for n in m.graph.node if any(i=='' for i in n.input)]
assert not (forbidden & set(ops))
assert not empty
assert onnx_path.stat().st_size < 1440000
with zipfile.ZipFile('submission.zip','w',zipfile.ZIP_DEFLATED) as z:
    z.write(onnx_path, arcname=f'{TASK_ID}.onnx')
print('wrote submission.zip')

wrote submission.zip
